In [15]:
!pip -q install pandas numpy cryptography


In [16]:
import json, glob, os
import pandas as pd

json_paths = sorted(glob.glob("/content/all_data_20250817_*.json"))
json_paths


['/content/all_data_20250817_095650.json',
 '/content/all_data_20250817_095811.json',
 '/content/all_data_20250817_095959.json',
 '/content/all_data_20250817_100018.json']

In [17]:
import json, glob, os
import pandas as pd

json_paths = sorted(glob.glob("/content/all_data_20250817_*.json"))
assert json_paths, "No all_data_*.json found in /content. Upload the exports."

def load_outputs(path):
    with open(path, "r") as f:
        data = json.load(f)
    outputs = data.get("outputs", [])
    rows = []
    for o in outputs:
        pdct = o.get("parsed_data") or {}
        q = pdct.get("quantum_sentiment")
        if q:
            rows.append({
                "source_file": os.path.basename(path),
                "script_name": o.get("script_name"),
                "timestamp": q.get("timestamp") or o.get("timestamp"),
                "cycle": q.get("cycle") or o.get("cycle"),
                "classical_host": q.get("classical_host"),
                "classical_mate": q.get("classical_mate"),
                "classical_shared": q.get("classical_shared"),
                "quantum_host": q.get("quantum_host"),
                "quantum_mate": q.get("quantum_mate"),
                "quantum_shared": q.get("quantum_shared"),
            })
    return pd.DataFrame(rows)

telemetry = pd.concat([load_outputs(p) for p in json_paths], ignore_index=True)
telemetry["timestamp"] = pd.to_datetime(telemetry["timestamp"], errors="coerce")
telemetry["cycle"] = pd.to_numeric(telemetry["cycle"], errors="coerce").astype("Int64")
telemetry = telemetry.sort_values(["timestamp","script_name","cycle"], na_position="last").reset_index(drop=True)

telemetry.head(), telemetry.shape

(                     source_file script_name                  timestamp  \
 0  all_data_20250817_095650.json       model 2025-08-17 09:56:43.222236   
 1  all_data_20250817_095650.json      cookie 2025-08-17 09:56:43.749270   
 2  all_data_20250817_095650.json      client 2025-08-17 09:56:44.239768   
 3  all_data_20250817_095650.json       model 2025-08-17 09:56:45.642759   
 4  all_data_20250817_095650.json       model 2025-08-17 09:56:45.643461   
 
    cycle  classical_host  classical_mate  classical_shared  quantum_host  \
 0      1           0.714           0.733             0.709         0.376   
 1      1           0.562           0.710             0.603         0.272   
 2      1           0.468           0.714             0.349         0.288   
 3      1           0.714           0.733             0.709         0.376   
 4      2           0.714           0.733             0.709         0.376   
 
    quantum_mate  quantum_shared  
 0         0.328           0.240  
 1      

In [18]:
import numpy as np
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.backends import default_backend

def key_from_row(row, salt=b"hive-v1", info=b"sentiment-key"):
    # robust quantization: map floats -> int16 bytes deterministically
    vec = np.array([row["quantum_host"], row["quantum_mate"], row["quantum_shared"]], dtype=np.float64)
    if np.any(pd.isna(vec)):
        return None
    # clip to [-1,1] then scale
    vec = np.clip(vec, -1.0, 1.0)
    q = (vec * 32767.0).round().astype(np.int16)
    ikm = q.tobytes() + str(row.get("cycle")).encode()  # include cycle for uniqueness

    hkdf = HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=salt,
        info=info,
        backend=default_backend()
    )
    return hkdf.derive(ikm)

# attach keys for rows that have quantum values
telemetry["key32"] = telemetry.apply(key_from_row, axis=1)
telemetry[telemetry["key32"].notna()].head(5)


,source_file,script_name,timestamp,cycle,classical_host,classical_mate,classical_shared,quantum_host,quantum_mate,quantum_shared,key32
0,all_data_20250817_095650.json,model,2025-08-17 09:56:43.222236,1,0.714,0.733,0.709,0.376,0.328,0.240,b'\x1cD\xd2\xb5\x9c\xf2M\xaa\xd6\xe3\x18uu\xc5...
1,all_data_20250817_095650.json,cookie,2025-08-17 09:56:43.749270,1,0.562,0.710,0.603,0.272,0.288,0.216,b' B~Fk\xbb\xe5\xafo=\xb8\x1e\xb9\x04ipI0\xc1}...
2,all_data_20250817_095650.json,client,2025-08-17 09:56:44.239768,1,0.468,0.714,0.349,0.288,0.312,0.224,b'\xf0\xe5k\x8c\xc3\xf2\x9c\xef\xe5\xdf\x82\x0...
3,all_data_20250817_095650.json,model,2025-08-17 09:56:45.642759,1,0.714,0.733,0.709,0.376,0.328,0.240,b'\x1cD\xd2\xb5\x9c\xf2M\xaa\xd6\xe3\x18uu\xc5...
4,all_data_20250817_095650.json,model,2025-08-17 09:56:45.643461,2,0.714,0.733,0.709,0.376,0.328,0.240,b'_D\x90\xba3\x9e>\x04\xd3\xdfJ\xfc\xe7\x9b\x1...


In [19]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import os, base64

def encrypt_with_row(row, plaintext: bytes, aad: bytes = b"hive"):
    key = row["key32"]
    if key is None:
        raise ValueError("Row has no key")
    aesgcm = AESGCM(key)
    nonce = os.urandom(12)
    ct = aesgcm.encrypt(nonce, plaintext, aad)
    return {
        "nonce_b64": base64.b64encode(nonce).decode(),
        "ct_b64": base64.b64encode(ct).decode(),
        "cycle": int(row["cycle"]) if pd.notna(row["cycle"]) else None,
        "timestamp": str(row["timestamp"]),
        "script_name": row["script_name"],
    }

def decrypt_with_row(row, payload, aad: bytes = b"hive"):
    key = row["key32"]
    aesgcm = AESGCM(key)
    nonce = base64.b64decode(payload["nonce_b64"])
    ct = base64.b64decode(payload["ct_b64"])
    return aesgcm.decrypt(nonce, ct, aad)

# pick a row that has a key
row = telemetry[telemetry["key32"].notna()].iloc[0]
payload = encrypt_with_row(row, b"hello hive: quantum-locked message")
payload


{'nonce_b64': 'O7bmldkODUdLzMIM',
 'ct_b64': 'hwXjg8QTeLGty4giejGHoidHQoHxKDPvB3zaaX06QYEvcMmFHQLfFXpeTaySpOU8igM=',
 'cycle': 1,
 'timestamp': '2025-08-17 09:56:43.222236',
 'script_name': 'model'}

In [20]:
import sqlite3, hashlib, json
from pathlib import Path

db_path = "/content/collected_data.db"
assert Path(db_path).exists(), "Upload collected_data.db into /content first"

con = sqlite3.connect(db_path)
cur = con.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS hive_telemetry (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  timestamp TEXT,
  cycle INTEGER,
  script_name TEXT,
  source_file TEXT,
  classical_host REAL,
  classical_mate REAL,
  classical_shared REAL,
  quantum_host REAL,
  quantum_mate REAL,
  quantum_shared REAL,
  key_sha256 TEXT
)
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS hive_messages (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  created_at TEXT,
  cycle INTEGER,
  script_name TEXT,
  aad TEXT,
  nonce_b64 TEXT,
  ct_b64 TEXT,
  key_sha256 TEXT,
  meta_json TEXT
)
""")

cur.execute("CREATE INDEX IF NOT EXISTS idx_tel_cycle ON hive_telemetry(cycle)")
cur.execute("CREATE INDEX IF NOT EXISTS idx_msg_cycle ON hive_messages(cycle)")
con.commit()

print("DB ready:", db_path)


DB ready: /content/collected_data.db


In [21]:
import pandas as pd

def sha256_hex(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

tel = telemetry.copy()
tel = tel[tel["key32"].notna()].copy()
tel["key_sha256"] = tel["key32"].apply(lambda k: sha256_hex(k))

# write to db (append)
cols = [
    "timestamp","cycle","script_name","source_file",
    "classical_host","classical_mate","classical_shared",
    "quantum_host","quantum_mate","quantum_shared",
    "key_sha256"
]
tel_to_write = tel[cols].copy()
tel_to_write["timestamp"] = tel_to_write["timestamp"].astype(str)

tel_to_write.to_sql("hive_telemetry", con, if_exists="append", index=False)

con.commit()
print("Inserted telemetry rows:", len(tel_to_write))


Inserted telemetry rows: 49


In [22]:
pd.read_sql_query("SELECT cycle, script_name, timestamp, key_sha256 FROM hive_telemetry ORDER BY id DESC LIMIT 10", con)


,cycle,script_name,timestamp,key_sha256
0,1,blockheart,2025-08-17 10:00:17.212964,f9d1ecf7b4dd60af071665f49a5768740960141d7a665b...
1,1,blockheart,2025-08-17 10:00:17.127296,f9d1ecf7b4dd60af071665f49a5768740960141d7a665b...
2,1,model,2025-08-17 10:00:16.397031,ca1a6f12aad5c1e27eb6e48b332f6e0b58eaa5026d4c36...
3,1,brian,2025-08-17 10:00:15.636896,e6fbbb78e882abc6fb1e3d5c76dcc5c25ab9d007663439...
4,4,cookie,2025-08-17 09:56:49.464948,ede290ab19e953a870f7f026c3b0619ca6d6772438132c...
5,4,cookie,2025-08-17 09:56:49.464790,ede290ab19e953a870f7f026c3b0619ca6d6772438132c...
6,4,cookie,2025-08-17 09:56:49.384530,ede290ab19e953a870f7f026c3b0619ca6d6772438132c...
7,4,cookie,2025-08-17 09:56:49.384412,ecbd04bcdc429120a2133726b1017082597920a3454627...
8,4,cookie,2025-08-17 09:56:49.384299,ecbd04bcdc429120a2133726b1017082597920a3454627...
9,4,cookie,2025-08-17 09:56:49.384174,ecbd04bcdc429120a2133726b1017082597920a3454627...


In [23]:
from datetime import datetime
import base64

def store_message(row, plaintext: bytes, aad: bytes = b"hive"):
    payload = encrypt_with_row(row, plaintext, aad=aad)

    key_sha = sha256_hex(row["key32"])
    meta = {
        "source_file": row.get("source_file"),
        "ts": str(row.get("timestamp")),
        "classical": {
            "host": float(row.get("classical_host")),
            "mate": float(row.get("classical_mate")),
            "shared": float(row.get("classical_shared")),
        },
        "quantum": {
            "host": float(row.get("quantum_host")),
            "mate": float(row.get("quantum_mate")),
            "shared": float(row.get("quantum_shared")),
        }
    }

    cur.execute("""
      INSERT INTO hive_messages (created_at, cycle, script_name, aad, nonce_b64, ct_b64, key_sha256, meta_json)
      VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        datetime.utcnow().isoformat(timespec="seconds") + "Z",
        int(row["cycle"]) if pd.notna(row["cycle"]) else None,
        str(row["script_name"]),
        aad.decode("utf-8", errors="replace"),
        payload["nonce_b64"],
        payload["ct_b64"],
        key_sha,
        json.dumps(meta, ensure_ascii=False),
    ))
    con.commit()
    return payload

# pick a row (you can choose a specific cycle later)
row = telemetry[telemetry["key32"].notna()].iloc[0]
payload = store_message(row, b"hello hive: stored in sqlite")
payload


/tmp/ipython-input-2367696085.py:27: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow().isoformat(timespec="seconds") + "Z",


{'nonce_b64': 'K5lPTZXGUjMayCur',
 'ct_b64': 'K7R5RbTHBhpuqv4x9f+xfqF+iam5kJBH7PAo2hrKa1MainOnTdmTiJNFKcI=',
 'cycle': 1,
 'timestamp': '2025-08-17 09:56:43.222236',
 'script_name': 'model'}

In [26]:
pd.read_sql_query("SELECT id, created_at, cycle, script_name, aad, key_sha, key_kind FROM hive_messages ORDER BY id DESC LIMIT 5", con)

DatabaseError: Execution failed on sql 'SELECT id, created_at, cycle, script_name, aad, key_sha, key_kind FROM hive_messages ORDER BY id DESC LIMIT 5': no such column: key_sha

In [27]:
def load_latest_message():
    df = pd.read_sql_query("SELECT * FROM hive_messages ORDER BY id DESC LIMIT 1", con)
    return df.iloc[0].to_dict()

msg = load_latest_message()
msg


{'id': 1,
 'created_at': '2026-01-12T19:22:40Z',
 'cycle': 1,
 'script_name': 'model',
 'aad': 'hive',
 'nonce_b64': 'K5lPTZXGUjMayCur',
 'ct_b64': 'K7R5RbTHBhpuqv4x9f+xfqF+iam5kJBH7PAo2hrKa1MainOnTdmTiJNFKcI=',
 'key_sha256': '34a65f0ff239b21a299084769984c3d06cf3f6b068b4437254b3c3375970b67c',
 'meta_json': '{"source_file": "all_data_20250817_095650.json", "ts": "2025-08-17 09:56:43.222236", "classical": {"host": 0.714, "mate": 0.733, "shared": 0.709}, "quantum": {"host": 0.376, "mate": 0.328, "shared": 0.24}}'}

In [28]:
cycle = msg["cycle"]
script = msg["script_name"]

candidate = telemetry[
    (telemetry["cycle"] == cycle) &
    (telemetry["script_name"] == script) &
    (telemetry["key32"].notna())
].iloc[0]

pt = decrypt_with_row(candidate, msg, aad=msg["aad"].encode())
pt


b'hello hive: stored in sqlite'

In [29]:
!pip -q install cirq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.8/670.8 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 430.5/430.5 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 53.0 MB/s eta 0:00:00


In [30]:
import cirq
import numpy as np

def circuit_projection_bits(row, n_qubits=6, reps=256):
    # map floats [-1,1] -> angles
    v = np.array([row["quantum_host"], row["quantum_mate"], row["quantum_shared"]], dtype=float)
    v = np.clip(v, -1.0, 1.0)
    angles = (v + 1.0) * np.pi  # [0, 2π]

    qs = cirq.LineQubit.range(n_qubits)
    c = cirq.Circuit()

    # simple “sentiment embedding” across qubits
    for i,q in enumerate(qs):
        a = angles[i % 3]
        c.append(cirq.ry(a)(q))
        c.append(cirq.rz(a/2)(q))

    # light entanglement
    for i in range(n_qubits-1):
        c.append(cirq.CNOT(qs[i], qs[i+1]))

    c.append(cirq.measure(*qs, key="m"))

    sim = cirq.Simulator()
    res = sim.run(c, repetitions=reps)
    bits = res.measurements["m"]  # shape (reps, n_qubits)
    # compress to bytes deterministically
    packed = np.packbits(bits.astype(np.uint8), axis=1).tobytes()
    return packed  # bytes

def key_from_row_cirq(row, salt=b"hive-v1", info=b"cirq-proj"):
    ikm = circuit_projection_bits(row) + str(row.get("cycle")).encode()
    hkdf = HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=salt,
        info=info,
        backend=default_backend()
    )
    return hkdf.derive(ikm)

# Example: generate cirq-based key
row = telemetry[telemetry["key32"].notna()].iloc[0]
k_cirq = key_from_row_cirq(row)
hashlib.sha256(k_cirq).hexdigest()[:16]


'ead9c993f3c9828d'

In [31]:
!pip -q install pandas numpy cryptography scikit-learn cirq


In [32]:
import json, glob, os
import pandas as pd

json_paths = sorted(glob.glob("/content/all_data_20250817_*.json"))
assert json_paths, "No all_data_*.json found in /content. Upload the exports."

def load_outputs(path):
    with open(path, "r") as f:
        data = json.load(f)
    outputs = data.get("outputs", [])
    rows = []
    for o in outputs:
        pdct = o.get("parsed_data") or {}
        q = pdct.get("quantum_sentiment")
        if q:
            rows.append({
                "source_file": os.path.basename(path),
                "script_name": o.get("script_name"),
                "timestamp": q.get("timestamp") or o.get("timestamp"),
                "cycle": q.get("cycle") or o.get("cycle"),
                "classical_host": q.get("classical_host"),
                "classical_mate": q.get("classical_mate"),
                "classical_shared": q.get("classical_shared"),
                "quantum_host": q.get("quantum_host"),
                "quantum_mate": q.get("quantum_mate"),
                "quantum_shared": q.get("quantum_shared"),
            })
    return pd.DataFrame(rows)

telemetry = pd.concat([load_outputs(p) for p in json_paths], ignore_index=True)
telemetry["timestamp"] = pd.to_datetime(telemetry["timestamp"], errors="coerce")
telemetry["cycle"] = pd.to_numeric(telemetry["cycle"], errors="coerce").astype("Int64")
telemetry = telemetry.sort_values(["timestamp","script_name","cycle"], na_position="last").reset_index(drop=True)

telemetry.head(), telemetry.shape


(                     source_file script_name                  timestamp  \
 0  all_data_20250817_095650.json       model 2025-08-17 09:56:43.222236   
 1  all_data_20250817_095650.json      cookie 2025-08-17 09:56:43.749270   
 2  all_data_20250817_095650.json      client 2025-08-17 09:56:44.239768   
 3  all_data_20250817_095650.json       model 2025-08-17 09:56:45.642759   
 4  all_data_20250817_095650.json       model 2025-08-17 09:56:45.643461   
 
    cycle  classical_host  classical_mate  classical_shared  quantum_host  \
 0      1           0.714           0.733             0.709         0.376   
 1      1           0.562           0.710             0.603         0.272   
 2      1           0.468           0.714             0.349         0.288   
 3      1           0.714           0.733             0.709         0.376   
 4      2           0.714           0.733             0.709         0.376   
 
    quantum_mate  quantum_shared  
 0         0.328           0.240  
 1      

In [33]:
import numpy as np
import hashlib
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.backends import default_backend
import cirq

def sha256_hex(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def hkdf32(ikm: bytes, salt=b"hive-v1", info=b"sentiment-key") -> bytes:
    hkdf = HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=salt,
        info=info,
        backend=default_backend()
    )
    return hkdf.derive(ikm)

def key_from_row_simple(row, salt=b"hive-v1", info=b"simple-v1"):
    vec = np.array([row["quantum_host"], row["quantum_mate"], row["quantum_shared"]], dtype=np.float64)
    if np.any(pd.isna(vec)):
        return None
    vec = np.clip(vec, -1.0, 1.0)
    q = (vec * 32767.0).round().astype(np.int16)
    ikm = q.tobytes() + str(int(row["cycle"]) if pd.notna(row["cycle"]) else -1).encode()
    return hkdf32(ikm, salt=salt, info=info)

def circuit_projection_bytes(row, n_qubits=6, reps=256):
    v = np.array([row["quantum_host"], row["quantum_mate"], row["quantum_shared"]], dtype=float)
    if np.any(pd.isna(v)):
        return None
    v = np.clip(v, -1.0, 1.0)
    angles = (v + 1.0) * np.pi  # [0, 2π]

    qs = cirq.LineQubit.range(n_qubits)
    c = cirq.Circuit()

    for i,qb in enumerate(qs):
        a = angles[i % 3]
        c.append(cirq.ry(a)(qb))
        c.append(cirq.rz(a/2)(qb))
    for i in range(n_qubits-1):
        c.append(cirq.CNOT(qs[i], qs[i+1]))

    c.append(cirq.measure(*qs, key="m"))

    sim = cirq.Simulator()
    res = sim.run(c, repetitions=reps)
    bits = res.measurements["m"].astype(np.uint8)  # (reps, n_qubits)
    packed = np.packbits(bits, axis=1).tobytes()
    return packed

def key_from_row_cirq(row, salt=b"hive-v1", info=b"cirq-proj-v1"):
    packed = circuit_projection_bytes(row)
    if packed is None:
        return None
    ikm = packed + str(int(row["cycle"]) if pd.notna(row["cycle"]) else -1).encode()
    return hkdf32(ikm, salt=salt, info=info)

telemetry["key32_simple"] = telemetry.apply(key_from_row_simple, axis=1)
telemetry["key32_cirq"]   = telemetry.apply(key_from_row_cirq, axis=1)

telemetry["key_sha_simple"] = telemetry["key32_simple"].apply(lambda k: sha256_hex(k) if isinstance(k,(bytes,bytearray)) else None)
telemetry["key_sha_cirq"]   = telemetry["key32_cirq"].apply(lambda k: sha256_hex(k) if isinstance(k,(bytes,bytearray)) else None)

telemetry[telemetry["key_sha_cirq"].notna()].head()


,source_file,script_name,timestamp,cycle,classical_host,classical_mate,classical_shared,quantum_host,quantum_mate,quantum_shared,key32_simple,key32_cirq,key_sha_simple,key_sha_cirq
0,all_data_20250817_095650.json,model,2025-08-17 09:56:43.222236,1,0.714,0.733,0.709,0.376,0.328,0.240,b'\xb93\x9a\x08\x91\x1d\x13\xb64\')\xf1\x18\xf...,b'\x92\xbf)\xd9\xfcw9\xe9!\x0f^\xba]\x16IP\xa5...,4bc5209475d2bfdd7c11f99841e59755ed1624bbcd37e6...,1c3f3298946ec96f299a11809cd0f03d6f11e4197cca95...
1,all_data_20250817_095650.json,cookie,2025-08-17 09:56:43.749270,1,0.562,0.710,0.603,0.272,0.288,0.216,b'\xb6\x07.g\xa4\xf9L\xdb\x1f:\x04\x114\xe5\x1...,b'Bw.\xcd\r5c\xe6\xf9\xf1\x86\x00\x85\x1c\x9e\...,10e20666fd8fd70b1d1bd7baf01ecc3cdf86b3162b4edb...,ad2966f820c0db900d080a15751baeb9788108ad73d281...
2,all_data_20250817_095650.json,client,2025-08-17 09:56:44.239768,1,0.468,0.714,0.349,0.288,0.312,0.224,b'~\xaaG=\rUU\t\x8e\x057_\xc3\xe2\xebNQd3t\xfc...,b'\xf8\r\r\x9b\xd6#\xd8\xcc\xb7\x99M\x0b\xea\x...,dd201754f43ceff13d37ac555b7f55838320978487b434...,ede72c54b70df240d56f89450e469717665df16afa6300...
3,all_data_20250817_095650.json,model,2025-08-17 09:56:45.642759,1,0.714,0.733,0.709,0.376,0.328,0.240,b'\xb93\x9a\x08\x91\x1d\x13\xb64\')\xf1\x18\xf...,b'Y\x86\x1d^\x18\xa3>\xb3\xa6\x19\x98\xabl\xde...,4bc5209475d2bfdd7c11f99841e59755ed1624bbcd37e6...,358539cd6055b39ec5e0572c0a02af14076901e96c9bf6...
4,all_data_20250817_095650.json,model,2025-08-17 09:56:45.643461,2,0.714,0.733,0.709,0.376,0.328,0.240,b'`d\xaa\xb0\xb0\x9d\xa5\xb0\xcd\xf3sjI\xc6r\x...,b'\xe1T\xa5p\x80\xa2\x86\xf4\x9e\x86\xe92\x81\...,b66d4d66274f41dced77dd092582550091072aeb3ee84c...,2805fb4feeef661aa4661e3323ebd5710d2a48bc49bcee...


In [34]:
import sqlite3
from pathlib import Path

db_path = "/content/collected_data.db"
assert Path(db_path).exists(), "Upload collected_data.db into /content first"

con = sqlite3.connect(db_path)
cur = con.cursor()

# Drop the tables if they exist to ensure schema update
cur.execute("DROP TABLE IF EXISTS hive_telemetry")
cur.execute("DROP TABLE IF EXISTS hive_messages")

cur.execute("""
CREATE TABLE IF NOT EXISTS hive_telemetry (
  timestamp TEXT NOT NULL,
  cycle INTEGER,
  script_name TEXT NOT NULL,
  source_file TEXT,
  classical_host REAL,
  classical_mate REAL,
  classical_shared REAL,
  quantum_host REAL,
  quantum_mate REAL,
  quantum_shared REAL,
  key_sha_simple TEXT,
  key_sha_cirq TEXT,
  PRIMARY KEY (timestamp, script_name, cycle)
)
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS hive_messages (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  created_at TEXT NOT NULL,
  timestamp TEXT,
  cycle INTEGER,
  script_name TEXT,
  aad TEXT,
  nonce_b64 TEXT,
  ct_b64 TEXT,
  key_sha TEXT,
  key_kind TEXT,
  meta_json TEXT
)
""")

cur.execute("CREATE INDEX IF NOT EXISTS idx_tel_cycle ON hive_telemetry(cycle)")
cur.execute("CREATE INDEX IF NOT EXISTS idx_msg_cycle ON hive_messages(cycle)")
con.commit()

print("DB schema ready.")

DB schema ready.


In [35]:
tel = telemetry.copy()
tel = tel[tel["timestamp"].notna() & tel["script_name"].notna()].copy()

def to_py(v):
    if pd.isna(v): return None
    if isinstance(v, (pd.Timestamp,)): return v.isoformat()
    if isinstance(v, (pd._libs.missing.NAType,)): return None
    return v

rows = []
for _, r in tel.iterrows():
    rows.append((
        to_py(r["timestamp"]),
        int(r["cycle"]) if pd.notna(r["cycle"]) else None,
        str(r["script_name"]),
        str(r["source_file"]) if pd.notna(r["source_file"]) else None,
        to_py(r["classical_host"]), to_py(r["classical_mate"]), to_py(r["classical_shared"]),
        to_py(r["quantum_host"]),   to_py(r["quantum_mate"]),   to_py(r["quantum_shared"]),
        r["key_sha_simple"],
        r["key_sha_cirq"],
    ))

cur.executemany("""
INSERT INTO hive_telemetry (
  timestamp, cycle, script_name, source_file,
  classical_host, classical_mate, classical_shared,
  quantum_host, quantum_mate, quantum_shared,
  key_sha_simple, key_sha_cirq
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
ON CONFLICT(timestamp, script_name, cycle) DO UPDATE SET
  source_file=excluded.source_file,
  classical_host=excluded.classical_host,
  classical_mate=excluded.classical_mate,
  classical_shared=excluded.classical_shared,
  quantum_host=excluded.quantum_host,
  quantum_mate=excluded.quantum_mate,
  quantum_shared=excluded.quantum_shared,
  key_sha_simple=excluded.key_sha_simple,
  key_sha_cirq=excluded.key_sha_cirq
""", rows)

con.commit()
print("Upserted telemetry rows:", len(rows))

Upserted telemetry rows: 49


In [36]:
import os, base64, json
from datetime import datetime
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

def encrypt_with_key(key32: bytes, plaintext: bytes, aad: bytes=b"hive"):
    aesgcm = AESGCM(key32)
    nonce = os.urandom(12)
    ct = aesgcm.encrypt(nonce, plaintext, aad)
    return base64.b64encode(nonce).decode(), base64.b64encode(ct).decode()

def decrypt_with_key(key32: bytes, nonce_b64: str, ct_b64: str, aad: bytes=b"hive"):
    aesgcm = AESGCM(key32)
    nonce = base64.b64decode(nonce_b64)
    ct = base64.b64decode(ct_b64)
    return aesgcm.decrypt(nonce, ct, aad)

def store_message(row, plaintext: bytes, key_kind="cirq", aad: bytes=b"hive"):
    key32 = row["key32_cirq"] if key_kind == "cirq" else row["key32_simple"]
    if not isinstance(key32, (bytes, bytearray)):
        raise ValueError("Row has no key for kind=" + key_kind)

    nonce_b64, ct_b64 = encrypt_with_key(key32, plaintext, aad=aad)
    key_sha = sha256_hex(key32)

    meta = {
        "source_file": row.get("source_file"),
        "ts": str(row.get("timestamp")),
        "classical": {
            "host": float(row.get("classical_host")) if pd.notna(row.get("classical_host")) else None,
            "mate": float(row.get("classical_mate")) if pd.notna(row.get("classical_mate")) else None,
            "shared": float(row.get("classical_shared")) if pd.notna(row.get("classical_shared")) else None,
        },
        "quantum": {
            "host": float(row.get("quantum_host")) if pd.notna(row.get("quantum_host")) else None,
            "mate": float(row.get("quantum_mate")) if pd.notna(row.get("quantum_mate")) else None,
            "shared": float(row.get("quantum_shared")) if pd.notna(row.get("quantum_shared")) else None,
        }
    }

    cur.execute("""
      INSERT INTO hive_messages (created_at, timestamp, cycle, script_name, aad, nonce_b64, ct_b64, key_sha, key_kind, meta_json)
      VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        datetime.utcnow().isoformat(timespec="seconds") + "Z",
        str(row["timestamp"]),
        int(row["cycle"]) if pd.notna(row["cycle"]) else None,
        str(row["script_name"]),
        aad.decode("utf-8", errors="replace"),
        nonce_b64, ct_b64,
        key_sha, key_kind,
        json.dumps(meta, ensure_ascii=False),
    ))
    con.commit()
    return nonce_b64, ct_b64, key_sha

row0 = telemetry[telemetry["key32_cirq"].notna()].iloc[0]
nonce_b64, ct_b64, key_sha = store_message(row0, b"hello hive: cirq-locked message", key_kind="cirq")
(key_sha, nonce_b64[:10], ct_b64[:10])

/tmp/ipython-input-2248142252.py:44: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow().isoformat(timespec="seconds") + "Z",


('1c3f3298946ec96f299a11809cd0f03d6f11e4197cca95ba5027fb9be087f831',
 'TGu9Th/SFL',
 'D/3Vk+IBSg')

In [37]:
import os, base64, json
from datetime import datetime
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

def encrypt_with_key(key32: bytes, plaintext: bytes, aad: bytes=b"hive"):
    aesgcm = AESGCM(key32)
    nonce = os.urandom(12)
    ct = aesgcm.encrypt(nonce, plaintext, aad)
    return base64.b64encode(nonce).decode(), base64.b64encode(ct).decode()

def decrypt_with_key(key32: bytes, nonce_b64: str, ct_b64: str, aad: bytes=b"hive"):
    aesgcm = AESGCM(key32)
    nonce = base64.b64decode(nonce_b64)
    ct = base64.b64decode(ct_b64)
    return aesgcm.decrypt(nonce, ct, aad)

def store_message(row, plaintext: bytes, key_kind="cirq", aad: bytes=b"hive"):
    key32 = row["key32_cirq"] if key_kind == "cirq" else row["key32_simple"]
    if not isinstance(key32, (bytes, bytearray)):
        raise ValueError("Row has no key for kind=" + key_kind)

    nonce_b64, ct_b64 = encrypt_with_key(key32, plaintext, aad=aad)
    key_sha = sha256_hex(key32)

    meta = {
        "source_file": row.get("source_file"),
        "ts": str(row.get("timestamp")),
        "classical": {
            "host": float(row.get("classical_host")) if pd.notna(row.get("classical_host")) else None,
            "mate": float(row.get("classical_mate")) if pd.notna(row.get("classical_mate")) else None,
            "shared": float(row.get("classical_shared")) if pd.notna(row.get("classical_shared")) else None,
        },
        "quantum": {
            "host": float(row.get("quantum_host")) if pd.notna(row.get("quantum_host")) else None,
            "mate": float(row.get("quantum_mate")) if pd.notna(row.get("quantum_mate")) else None,
            "shared": float(row.get("quantum_shared")) if pd.notna(row.get("quantum_shared")) else None,
        }
    }

    cur.execute("""
      INSERT INTO hive_messages (created_at, timestamp, cycle, script_name, aad, nonce_b64, ct_b64, key_sha, key_kind, meta_json)
      VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        datetime.utcnow().isoformat(timespec="seconds") + "Z",
        str(row["timestamp"]),
        int(row["cycle"]) if pd.notna(row["cycle"]) else None,
        str(row["script_name"]),
        aad.decode("utf-8", errors="replace"),
        nonce_b64, ct_b64,
        key_sha, key_kind,
        json.dumps(meta, ensure_ascii=False),
    ))
    con.commit()
    return nonce_b64, ct_b64, key_sha

row0 = telemetry[telemetry["key32_cirq"].notna()].iloc[0]
nonce_b64, ct_b64, key_sha = store_message(row0, b"hello hive: cirq-locked message", key_kind="cirq")
(key_sha, nonce_b64[:10], ct_b64[:10])


/tmp/ipython-input-330299270.py:44: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow().isoformat(timespec="seconds") + "Z",


('1c3f3298946ec96f299a11809cd0f03d6f11e4197cca95ba5027fb9be087f831',
 'DRo2D6dkO2',
 'UCBxre4F9v')

In [38]:
import numpy as np

df = telemetry.copy()
df = df[df["timestamp"].notna()].copy()

# Basic features
df["hour"] = df["timestamp"].dt.hour.astype(float)
df["minute"] = df["timestamp"].dt.minute.astype(float)

# Target(s)
targets = ["quantum_host","quantum_mate","quantum_shared"]

# Keep rows where we have classical + quantum
needed = ["classical_host","classical_mate","classical_shared"] + targets
df = df.dropna(subset=needed + ["script_name"])

# One-hot encode script_name
X = df[["classical_host","classical_mate","classical_shared","hour","minute"]].copy()
X = pd.concat([X, pd.get_dummies(df["script_name"], prefix="script")], axis=1)

y_shared = df["quantum_shared"].astype(float).values
y_host   = df["quantum_host"].astype(float).values
y_mate   = df["quantum_mate"].astype(float).values

X.shape, df.shape


((49, 10), (49, 16))

In [39]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

X_train, X_test, y_train, y_test = train_test_split(X, y_shared, test_size=0.2, random_state=42)

reg = RandomForestRegressor(
    n_estimators=400,
    random_state=42,
    n_jobs=-1
)
reg.fit(X_train, y_train)
pred = reg.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))
print("R2 :", r2_score(y_test, pred))


MAE: 0.010541086652236511
R2 : -0.6333787491091676


In [40]:
from sklearn.ensemble import IsolationForest

feat_cols = ["classical_host","classical_mate","classical_shared","quantum_host","quantum_mate","quantum_shared"]
A = df[feat_cols].astype(float).values

iso = IsolationForest(n_estimators=400, contamination=0.03, random_state=42)
scores = iso.fit_predict(A)  # -1 anomaly, +1 normal
df["anomaly"] = (scores == -1)

df[df["anomaly"]].head(20)[["timestamp","cycle","script_name"] + feat_cols]


,timestamp,cycle,script_name,classical_host,classical_mate,classical_shared,quantum_host,quantum_mate,quantum_shared
32,2025-08-17 09:56:48.403926,3,client,0.786,0.485,0.733,0.456,0.344,0.344
36,2025-08-17 09:56:49.019539,4,model,0.633,0.588,0.293,0.424,0.416,0.208


In [41]:
df.groupby("script_name")["anomaly"].mean().sort_values(ascending=False)


,anomaly
script_name,
client,0.090909
model,0.058824
blockheart,0.000000
brian,0.000000
cookie,0.000000


In [42]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

y_script = df["script_name"].values
Xc = df[feat_cols + ["hour","minute"]].astype(float)
Xc = (Xc - Xc.mean()) / (Xc.std() + 1e-9)  # standardize

# Check the distribution of classes in y_script
print("Script name counts:\n", df["script_name"].value_counts())

# Filter out script names with only one occurrence
script_counts = df["script_name"].value_counts()
scripts_to_keep = script_counts[script_counts > 1].index
df_filtered = df[df["script_name"].isin(scripts_to_keep)]

y_script_filtered = df_filtered["script_name"].values
Xc_filtered = Xc[df["script_name"].isin(scripts_to_keep)]

X_train, X_test, y_train, y_test = train_test_split(Xc_filtered, y_script_filtered, test_size=0.2, random_state=42, stratify=y_script_filtered)

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Script name counts:
 script_name
cookie        18
model         17
client        11
blockheart     2
brian          1
Name: count, dtype: int64
Accuracy: 0.8
              precision    recall  f1-score   support

      client       0.00      0.00      0.00         2
      cookie       0.67      1.00      0.80         4
       model       1.00      1.00      1.00         4

    accuracy                           0.80        10
   macro avg       0.56      0.67      0.60        10
weighted avg       0.67      0.80      0.72        10



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [45]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

y_script = df["script_name"].values
Xc = df[feat_cols + ["hour","minute"]].astype(float)
Xc = (Xc - Xc.mean()) / (Xc.std() + 1e-9)  # standardize

# Filter out script names with only one occurrence to allow for stratified splitting
script_counts = df["script_name"].value_counts()
scripts_to_keep = script_counts[script_counts > 1].index
df_filtered = df[df["script_name"].isin(scripts_to_keep)]

y_script_filtered = df_filtered["script_name"].values
Xc_filtered = Xc[df["script_name"].isin(scripts_to_keep)]

X_train, X_test, y_train, y_test = train_test_split(Xc_filtered, y_script_filtered, test_size=0.2, random_state=42, stratify=y_script_filtered)

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.8
              precision    recall  f1-score   support

      client       0.00      0.00      0.00         2
      cookie       0.67      1.00      0.80         4
       model       1.00      1.00      1.00         4

    accuracy                           0.80        10
   macro avg       0.56      0.67      0.60        10
weighted avg       0.67      0.80      0.72        10



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [46]:
print("rows:", len(telemetry))
print("keys(simple):", telemetry["key32_simple"].notna().sum())
print("keys(cirq):", telemetry["key32_cirq"].notna().sum())
print("unique scripts:", telemetry["script_name"].nunique())
print("scripts:", sorted(telemetry["script_name"].dropna().unique())[:20])


rows: 49
keys(simple): 49
keys(cirq): 49
unique scripts: 5
scripts: ['blockheart', 'brian', 'client', 'cookie', 'model']


In [47]:
!pip -q install pandas numpy scikit-learn torch torchvision torchaudio

# Cirq + qsimcirq can be finicky; this usually works in Colab:
!pip -q install cirq qsimcirq

# Brian2
!pip -q install brian2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 539.3/539.3 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.8 MB/s eta 0:00:00


In [48]:
import json, glob, os
import pandas as pd
import numpy as np

json_paths = sorted(glob.glob("/content/all_data_20250817_*.json"))
assert json_paths, "Upload all_data_20250817_*.json files to /content."

def load_outputs(path):
    with open(path, "r") as f:
        data = json.load(f)
    outputs = data.get("outputs", [])
    rows = []
    for o in outputs:
        pdct = o.get("parsed_data") or {}
        q = pdct.get("quantum_sentiment")
        if q:
            rows.append({
                "source_file": os.path.basename(path),
                "script_name": o.get("script_name"),
                "timestamp": q.get("timestamp") or o.get("timestamp"),
                "cycle": q.get("cycle") or o.get("cycle"),
                "classical_host": q.get("classical_host"),
                "classical_mate": q.get("classical_mate"),
                "classical_shared": q.get("classical_shared"),
                "quantum_host": q.get("quantum_host"),
                "quantum_mate": q.get("quantum_mate"),
                "quantum_shared": q.get("quantum_shared"),
            })
    return pd.DataFrame(rows)

telemetry = pd.concat([load_outputs(p) for p in json_paths], ignore_index=True)
telemetry["timestamp"] = pd.to_datetime(telemetry["timestamp"], errors="coerce")
telemetry["cycle"] = pd.to_numeric(telemetry["cycle"], errors="coerce")
telemetry = telemetry.dropna(subset=["timestamp","script_name","cycle",
                                     "classical_host","classical_mate","classical_shared",
                                     "quantum_host","quantum_mate","quantum_shared"]).copy()
telemetry = telemetry.sort_values(["timestamp","script_name","cycle"]).reset_index(drop=True)

print("rows:", len(telemetry), "scripts:", telemetry["script_name"].nunique())
telemetry.head()


rows: 49 scripts: 5


,source_file,script_name,timestamp,cycle,classical_host,classical_mate,classical_shared,quantum_host,quantum_mate,quantum_shared
0,all_data_20250817_095650.json,model,2025-08-17 09:56:43.222236,1,0.714,0.733,0.709,0.376,0.328,0.240
1,all_data_20250817_095650.json,cookie,2025-08-17 09:56:43.749270,1,0.562,0.710,0.603,0.272,0.288,0.216
2,all_data_20250817_095650.json,client,2025-08-17 09:56:44.239768,1,0.468,0.714,0.349,0.288,0.312,0.224
3,all_data_20250817_095650.json,model,2025-08-17 09:56:45.642759,1,0.714,0.733,0.709,0.376,0.328,0.240
4,all_data_20250817_095650.json,model,2025-08-17 09:56:45.643461,2,0.714,0.733,0.709,0.376,0.328,0.240


In [49]:
import json, glob, os
import pandas as pd
import numpy as np

json_paths = sorted(glob.glob("/content/all_data_20250817_*.json"))
assert json_paths, "Upload all_data_20250817_*.json files to /content."

def load_outputs(path):
    with open(path, "r") as f:
        data = json.load(f)
    outputs = data.get("outputs", [])
    rows = []
    for o in outputs:
        pdct = o.get("parsed_data") or {}
        q = pdct.get("quantum_sentiment")
        if q:
            rows.append({
                "source_file": os.path.basename(path),
                "script_name": o.get("script_name"),
                "timestamp": q.get("timestamp") or o.get("timestamp"),
                "cycle": q.get("cycle") or o.get("cycle"),
                "classical_host": q.get("classical_host"),
                "classical_mate": q.get("classical_mate"),
                "classical_shared": q.get("classical_shared"),
                "quantum_host": q.get("quantum_host"),
                "quantum_mate": q.get("quantum_mate"),
                "quantum_shared": q.get("quantum_shared"),
            })
    return pd.DataFrame(rows)

telemetry = pd.concat([load_outputs(p) for p in json_paths], ignore_index=True)
telemetry["timestamp"] = pd.to_datetime(telemetry["timestamp"], errors="coerce")
telemetry["cycle"] = pd.to_numeric(telemetry["cycle"], errors="coerce")
telemetry = telemetry.dropna(subset=["timestamp","script_name","cycle",
                                     "classical_host","classical_mate","classical_shared",
                                     "quantum_host","quantum_mate","quantum_shared"]).copy()
telemetry = telemetry.sort_values(["timestamp","script_name","cycle"]).reset_index(drop=True)

print("rows:", len(telemetry), "scripts:", telemetry["script_name"].nunique())
telemetry.head()


rows: 49 scripts: 5


,source_file,script_name,timestamp,cycle,classical_host,classical_mate,classical_shared,quantum_host,quantum_mate,quantum_shared
0,all_data_20250817_095650.json,model,2025-08-17 09:56:43.222236,1,0.714,0.733,0.709,0.376,0.328,0.240
1,all_data_20250817_095650.json,cookie,2025-08-17 09:56:43.749270,1,0.562,0.710,0.603,0.272,0.288,0.216
2,all_data_20250817_095650.json,client,2025-08-17 09:56:44.239768,1,0.468,0.714,0.349,0.288,0.312,0.224
3,all_data_20250817_095650.json,model,2025-08-17 09:56:45.642759,1,0.714,0.733,0.709,0.376,0.328,0.240
4,all_data_20250817_095650.json,model,2025-08-17 09:56:45.643461,2,0.714,0.733,0.709,0.376,0.328,0.240


In [50]:
import cirq

# Try qsimcirq (fast). Fallback to cirq.Simulator if not available.
try:
    import qsimcirq
    HAVE_QSIM = True
except Exception:
    HAVE_QSIM = False

def make_circuit_from_row(row, n_qubits=6):
    v = np.array([row["quantum_host"], row["quantum_mate"], row["quantum_shared"]], dtype=float)
    v = np.clip(v, -1.0, 1.0)
    angles = (v + 1.0) * np.pi  # [0, 2π]

    qs = cirq.LineQubit.range(n_qubits)
    c = cirq.Circuit()

    for i, qb in enumerate(qs):
        a = angles[i % 3]
        c.append(cirq.ry(a)(qb))
        c.append(cirq.rz(a/2)(qb))
    for i in range(n_qubits-1):
        c.append(cirq.CNOT(qs[i], qs[i+1]))
    c.append(cirq.measure(*qs, key="m"))
    return c, qs

def sample_bits(circuit, reps=256):
    if HAVE_QSIM:
        sim = qsimcirq.QSimSimulator()
    else:
        sim = cirq.Simulator()
    res = sim.run(circuit, repetitions=reps)
    bits = res.measurements["m"].astype(np.uint8)  # (reps, n_qubits)
    return bits

def bits_to_hist(bits):
    # convert each measurement to integer 0..(2^n-1), then histogram
    reps, n = bits.shape
    vals = (bits * (2 ** np.arange(n)[None, :])).sum(axis=1)
    hist = np.bincount(vals, minlength=2**n).astype(np.float32)
    hist /= max(1, hist.sum())
    return hist

# quick test
r0 = telemetry.iloc[0].to_dict()
c0, _ = make_circuit_from_row(r0, n_qubits=6)
b0 = sample_bits(c0, reps=256)
f0 = bits_to_hist(b0)
print("feature dim:", f0.shape, "qsim:", HAVE_QSIM)


feature dim: (64,) qsim: True


In [51]:
from tqdm import tqdm

N_QUBITS = 6
REPS = 256
FEAT_DIM = 2**N_QUBITS

features = np.zeros((len(telemetry), FEAT_DIM), dtype=np.float32)

for i in tqdm(range(len(telemetry))):
    row = telemetry.iloc[i].to_dict()
    c, _ = make_circuit_from_row(row, n_qubits=N_QUBITS)
    bits = sample_bits(c, reps=REPS)
    features[i] = bits_to_hist(bits)

print("features:", features.shape)


100%|██████████| 49/49 [00:00<00:00, 140.71it/s]

features: (49, 64)


In [52]:
from collections import defaultdict

telemetry = telemetry.reset_index(drop=True)
telemetry["script_id"] = telemetry["script_name"].astype("category").cat.codes
n_scripts = telemetry["script_id"].nunique()

# group indices by script
by_script = defaultdict(list)
for idx, sid in enumerate(telemetry["script_id"].values):
    by_script[int(sid)].append(idx)

SEQ_LEN = 5  # Reduced window length to allow sequence creation

X_seq = []
Y_next = []
S_seq = []

targets = telemetry[["quantum_host","quantum_mate","quantum_shared"]].values.astype(np.float32)

for sid, idxs in by_script.items():
    # ensure time order already sorted; idxs are in sorted order due to global sort by timestamp+script
    for j in range(0, len(idxs) - SEQ_LEN - 1):
        win = idxs[j:j+SEQ_LEN]
        nxt = idxs[j+SEQ_LEN]
        X_seq.append(features[win])             # (SEQ_LEN, FEAT_DIM)
        Y_next.append(targets[nxt])             # (3,)
        S_seq.append(sid)                       # script id (observer identity)

X_seq = np.stack(X_seq).astype(np.float32)
Y_next = np.stack(Y_next).astype(np.float32)
S_seq = np.array(S_seq, dtype=np.int64)

print("dataset:", X_seq.shape, Y_next.shape, "scripts:", n_scripts)

dataset: (28, 5, 64) (28, 3) scripts: 5


In [53]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class HiveDataset(Dataset):
    def __init__(self, X, Y, S):
        self.X = torch.from_numpy(X)   # (N, T, D)
        self.Y = torch.from_numpy(Y)   # (N, 3)
        self.S = torch.from_numpy(S)   # (N,)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i):
        return self.X[i], self.Y[i], self.S[i]

class TransformerPredictor(nn.Module):
    def __init__(self, feat_dim, n_scripts, d_model=256, nhead=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.script_emb = nn.Embedding(n_scripts, d_model)
        self.in_proj = nn.Linear(feat_dim, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4*d_model, dropout=dropout,
            batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.out = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 128),
            nn.GELU(),
            nn.Linear(128, 3)
        )

    def forward(self, x, sid):
        # x: (B,T,D)
        h = self.in_proj(x)
        h = h + self.script_emb(sid).unsqueeze(1)  # add observer embedding
        h = self.encoder(h)
        last = h[:, -1, :]
        return self.out(last)

torch_device = "cuda" if torch.cuda.is_available() else "cpu"
model = TransformerPredictor(FEAT_DIM, n_scripts).to(torch_device)
model

TransformerPredictor(
  (script_emb): Embedding(5, 256)
  (in_proj): Linear(in_features=64, out_features=256, bias=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=1024, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (out): Sequential(
    (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=256, out_features=128, bias=True)
    (2): GELU(approximate=

In [54]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(X_seq))
train_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=42)

ds_train = HiveDataset(X_seq[train_idx], Y_next[train_idx], S_seq[train_idx])
ds_val   = HiveDataset(X_seq[val_idx],   Y_next[val_idx],   S_seq[val_idx])

dl_train = DataLoader(ds_train, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
dl_val   = DataLoader(ds_val, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
loss_fn = nn.SmoothL1Loss()

def eval_loss():
    model.eval()
    tot, n = 0.0, 0
    with torch.no_grad():
        for x,y,sid in dl_val:
            x,y,sid = x.to(torch_device), y.to(torch_device), sid.to(torch_device)
            pred = model(x, sid)
            loss = loss_fn(pred, y)
            tot += float(loss) * x.size(0)
            n += x.size(0)
    return tot/n

for epoch in range(1, 11):
    model.train()
    tot, n = 0.0, 0
    for x,y,sid in dl_train:
        x,y,sid = x.to(torch_device), y.to(torch_device), sid.to(torch_device)
        opt.zero_grad(set_to_none=True)
        pred = model(x, sid)
        loss = loss_fn(pred, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tot += float(loss) * x.size(0)
        n += x.size(0)

    print(f"epoch {epoch:02d} train {tot/n:.5f}  val {eval_loss():.5f}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/tmp/ipython-input-742500075.py:38: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  tot += float(loss) * x.size(0)


epoch 01 train 0.06902  val 0.03509
epoch 02 train 0.03271  val 0.01328
epoch 03 train 0.01249  val 0.00402
epoch 04 train 0.00682  val 0.00883
epoch 05 train 0.00888  val 0.01158
epoch 06 train 0.00903  val 0.00834
epoch 07 train 0.00576  val 0.00444
epoch 08 train 0.00288  val 0.00301
epoch 09 train 0.00267  val 0.00326
epoch 10 train 0.00334  val 0.00354


In [55]:
from brian2 import *

def brian2_decode(pred_vec, duration_ms=200):
    # pred_vec: (3,) floats in [-1,1] roughly (your targets are around 0..1, but we clip)
    v = np.array(pred_vec, dtype=float)
    v = np.clip(v, -1.0, 1.0)

    start_scope()
    defaultclock.dt = 0.1*ms

    N = 64
    tau = 10*ms
    eqs = '''
    dv/dt = (-v + I)/tau : 1
    I : 1
    '''
    G = NeuronGroup(N, eqs, threshold='v>1', reset='v=0', method='euler')

    # map vector to current profile
    # (simple: base + weighted components)
    base = 0.6
    weights = np.linspace(0.5, 1.5, N)
    drive = base + weights*(0.2*v[0] + 0.2*v[1] + 0.2*v[2])
    G.I = drive

    M = SpikeMonitor(G)
    run(duration_ms*ms)

    # return spike count and rate estimate
    count = M.count[:]  # spikes per neuron
    rate_hz = count.mean() / (duration_ms/1000.0)
    return float(rate_hz), count

# demo: run decoder on a sample
model.eval()
x,y,sid = ds_val[0]
with torch.no_grad():
    pred = model(x.unsqueeze(0).to(torch_device), torch.tensor([int(sid)]).to(torch_device)).cpu().numpy()[0]

rate_hz, counts = brian2_decode(pred, duration_ms=200)
rate_hz, pred

WARNING    'v' is an internal variable of group 'neurongroup', but also exists in the run namespace with the value array([0.45038694, 0.43403021, 0.24631375]). The internal variable will be used. [brian2.groups.group.Group.resolve.resolution_conflict]


(0.0, array([0.45038694, 0.4340302 , 0.24631375], dtype=float32))

In [56]:
telemetry.groupby("script_name")["cycle"].count().sort_values(ascending=False)


,cycle
script_name,
cookie,18
model,17
client,11
blockheart,2
brian,1


In [57]:
!pip -q install torch torchvision torchaudio transformers sentence-transformers accelerate \
  opencv-python pillow librosa ffmpeg-python pandas numpy scikit-learn

# optional (if you want CLIP):
!pip -q install ftfy regex tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.0 MB/s eta 0:00:00


In [58]:
{"timestamp":"2025-08-17T09:56:43.222236","script":"model","cycle":1,
 "text":"operator note ...", "image_path":".../frame_0001.jpg",
 "audio_path":".../clip.wav", "video_path":".../clip.mp4"}


{'timestamp': '2025-08-17T09:56:43.222236',
 'script': 'model',
 'cycle': 1,
 'text': 'operator note ...',
 'image_path': '.../frame_0001.jpg',
 'audio_path': '.../clip.wav',
 'video_path': '.../clip.mp4'}

In [59]:
from sentence_transformers import SentenceTransformer
text_teacher = SentenceTransformer("all-MiniLM-L6-v2")  # small + good
text_teacher.eval()


WARNING    /usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
 [py.warnings]
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [60]:
import torch
from transformers import CLIPProcessor, CLIPModel
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_proc  = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [61]:
from transformers import WhisperProcessor, WhisperModel
whisper_model = WhisperModel.from_pretrained("openai/whisper-small")  # encoder+decoder, we use encoder
whisper_proc  = WhisperProcessor.from_pretrained("openai/whisper-small")
whisper_model.eval()


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

WhisperModel(
  (encoder): WhisperEncoder(
    (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
    (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
    (embed_positions): Embedding(1500, 768)
    (layers): ModuleList(
      (0-11): 12 x WhisperEncoderLayer(
        (self_attn): WhisperAttention(
          (k_proj): Linear(in_features=768, out_features=768, bias=False)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (out_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (activation_fn): GELUActivation()
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
        (final_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      

In [62]:
from transformers import WhisperProcessor, WhisperModel
whisper_model = WhisperModel.from_pretrained("openai/whisper-small")  # encoder+decoder, we use encoder
whisper_proc  = WhisperProcessor.from_pretrained("openai/whisper-small")
whisper_model.eval()


WhisperModel(
  (encoder): WhisperEncoder(
    (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
    (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
    (embed_positions): Embedding(1500, 768)
    (layers): ModuleList(
      (0-11): 12 x WhisperEncoderLayer(
        (self_attn): WhisperAttention(
          (k_proj): Linear(in_features=768, out_features=768, bias=False)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (out_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (activation_fn): GELUActivation()
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
        (final_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      

In [63]:
import numpy as np
from PIL import Image
import librosa
import cv2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
clip_model = clip_model.to(DEVICE)
whisper_model = whisper_model.to(DEVICE)

@torch.no_grad()
def embed_text(t: str):
    if not t:
        return None
    v = text_teacher.encode([t], normalize_embeddings=True)[0].astype(np.float32)
    return v  # (d,)

@torch.no_grad()
def embed_image(path: str):
    if not path:
        return None
    img = Image.open(path).convert("RGB")
    inputs = clip_proc(images=img, return_tensors="pt").to(DEVICE)
    feats = clip_model.get_image_features(**inputs)
    feats = torch.nn.functional.normalize(feats, dim=-1)
    return feats[0].cpu().numpy().astype(np.float32)  # (512,)

@torch.no_grad()
def embed_audio(path: str, sr=16000, max_sec=20):
    if not path:
        return None
    wav, _ = librosa.load(path, sr=sr, mono=True)
    wav = wav[: sr*max_sec]
    inputs = whisper_proc(wav, sampling_rate=sr, return_tensors="pt")
    input_features = inputs.input_features.to(DEVICE)  # (1, 80, frames)
    enc = whisper_model.encoder(input_features).last_hidden_state  # (1, T, d)
    # pool
    v = enc.mean(dim=1)[0]
    v = torch.nn.functional.normalize(v, dim=-1)
    return v.cpu().numpy().astype(np.float32)  # (d,)

@torch.no_grad()
def embed_video(path: str, n_frames=8):
    if not path:
        return None
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        return None
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs = np.linspace(0, max(0, frame_count-1), n_frames).astype(int)

    embs = []
    cur = 0
    want = set(idxs.tolist())
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i in want:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = Image.fromarray(frame)
            inputs = clip_proc(images=img, return_tensors="pt").to(DEVICE)
            feats = clip_model.get_image_features(**inputs)
            feats = torch.nn.functional.normalize(feats, dim=-1)
            embs.append(feats[0].cpu().numpy())
        i += 1
    cap.release()
    if not embs:
        return None
    v = np.mean(np.stack(embs).astype(np.float32), axis=0)
    v = v / (np.linalg.norm(v) + 1e-9)
    return v.astype(np.float32)  # (512,)


In [64]:
import torch
import torch.nn as nn

class MultiModalStudent(nn.Module):
    def __init__(self, d_model=256, nhead=8, num_layers=4,
                 d_text=384, d_img=512, d_vid=512, d_aud=768, d_telem=64,
                 n_scripts=16):
        super().__init__()
        self.script_emb = nn.Embedding(n_scripts, d_model)

        self.p_text  = nn.Linear(d_text, d_model)
        self.p_img   = nn.Linear(d_img, d_model)
        self.p_vid   = nn.Linear(d_vid, d_model)
        self.p_aud   = nn.Linear(d_aud, d_model)
        self.p_tel   = nn.Linear(d_telem, d_model)

        # token type embeddings: 0=TEL,1=TXT,2=IMG,3=AUD,4=VID
        self.type_emb = nn.Embedding(5, d_model)

        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4*d_model,
            dropout=0.1, batch_first=True, activation="gelu"
        )
        self.tr = nn.TransformerEncoder(enc, num_layers=num_layers)

        # heads
        self.next_q = nn.Linear(d_model, 3)  # predict next (quantum_host,mate,shared)

        # distill heads (optional): match teacher embeddings (project back)
        self.dist_text = nn.Linear(d_model, d_text)
        self.dist_img  = nn.Linear(d_model, d_img)
        self.dist_aud  = nn.Linear(d_model, d_aud)
        self.dist_vid  = nn.Linear(d_model, d_vid)

    def forward(self, tel_seq, txt_seq, img_seq, aud_seq, vid_seq, mask_seq, script_id):
        """
        Inputs are sequences over time T:
          tel_seq: (B,T,d_telem)
          txt_seq/img_seq/aud_seq/vid_seq: (B,T,d_mod) or zeros if missing
          mask_seq: (B,T,5) 1 if present else 0
        We expand each time step into up to 5 tokens and flatten to (B, T*5, d_model).
        """
        B,T,_ = tel_seq.shape

        # project each modality
        tel = self.p_tel(tel_seq) + self.type_emb(torch.zeros((B,T),dtype=torch.long,device=tel_seq.device))
        txt = self.p_text(txt_seq) + self.type_emb(torch.ones((B,T),dtype=torch.long,device=tel_seq.device))
        img = self.p_img(img_seq)  + self.type_emb(torch.full((B,T),2,dtype=torch.long,device=tel_seq.device))
        aud = self.p_aud(aud_seq)  + self.type_emb(torch.full((B,T),3,dtype=torch.long,device=tel_seq.device))
        vid = self.p_vid(vid_seq)  + self.type_emb(torch.full((B,T),4,dtype=torch.long,device=tel_seq.device))

        # add observer/script embedding to all tokens
        s = self.script_emb(script_id).unsqueeze(1).unsqueeze(1)  # (B,1,1,d)
        tel,txt,img,aud,vid = tel+s,txt+s,img+s,aud+s,vid+s

        # stack tokens per time: (B,T,5,d) -> (B,T*5,d)
        tokens = torch.stack([tel,txt,img,aud,vid], dim=2)
        tokens = tokens.reshape(B, T*5, -1)

        # attention mask: True means "ignore"
        present = mask_seq.reshape(B, T*5)  # 1 present, 0 absent
        src_key_padding_mask = (present == 0)

        h = self.tr(tokens, src_key_padding_mask=src_key_padding_mask)

        # use the last TELEMETRY token at final timestep as summary (index = (T-1)*5 + 0)
        idx = (T-1)*5 + 0
        summary = h[:, idx, :]

        next_q = self.next_q(summary)

        # distill: predict teacher embeddings for *this timestep summary* (you can also do per-modality token)
        return next_q, {
            "text": self.dist_text(summary),
            "img":  self.dist_img(summary),
            "aud":  self.dist_aud(summary),
            "vid":  self.dist_vid(summary),
        }


In [65]:
def cosine_loss(a, b, eps=1e-8):
    a = a / (a.norm(dim=-1, keepdim=True) + eps)
    b = b / (b.norm(dim=-1, keepdim=True) + eps)
    return 1.0 - (a*b).sum(dim=-1).mean()


In [66]:
!pip -q install pandas numpy scikit-learn torch torchvision torchaudio transformers sentence-transformers \
  opencv-python pillow librosa ffmpeg-python matplotlib tqdm

# Cirq + qsimcirq + Brian2
!pip -q install cirq qsimcirq brian2


In [67]:
import json, glob, os
import pandas as pd
import numpy as np

json_paths = sorted(glob.glob("/content/all_data_20250817_*.json"))
assert json_paths, "Upload all_data_20250817_*.json files to /content."

def load_outputs(path):
    with open(path, "r") as f:
        data = json.load(f)
    outputs = data.get("outputs", [])
    rows = []
    for o in outputs:
        pdct = o.get("parsed_data") or {}
        q = pdct.get("quantum_sentiment")
        if q:
            rows.append({
                "source_file": os.path.basename(path),
                "script_name": o.get("script_name"),
                "timestamp": q.get("timestamp") or o.get("timestamp"),
                "cycle": q.get("cycle") or o.get("cycle"),
                "classical_host": q.get("classical_host"),
                "classical_mate": q.get("classical_mate"),
                "classical_shared": q.get("classical_shared"),
                "quantum_host": q.get("quantum_host"),
                "quantum_mate": q.get("quantum_mate"),
                "quantum_shared": q.get("quantum_shared"),
            })
    return pd.DataFrame(rows)

telemetry = pd.concat([load_outputs(p) for p in json_paths], ignore_index=True)
telemetry["timestamp"] = pd.to_datetime(telemetry["timestamp"], errors="coerce")
telemetry["cycle"] = pd.to_numeric(telemetry["cycle"], errors="coerce")
telemetry = telemetry.dropna(subset=[
    "timestamp","script_name","cycle",
    "classical_host","classical_mate","classical_shared",
    "quantum_host","quantum_mate","quantum_shared"
]).copy()

telemetry = telemetry.sort_values(["timestamp","script_name","cycle"]).reset_index(drop=True)
telemetry["script_id"] = telemetry["script_name"].astype("category").cat.codes
n_scripts = telemetry["script_id"].nunique()

print("rows:", len(telemetry), "scripts:", n_scripts)
telemetry.head()


rows: 49 scripts: 5


,source_file,script_name,timestamp,cycle,classical_host,classical_mate,classical_shared,quantum_host,quantum_mate,quantum_shared,script_id
0,all_data_20250817_095650.json,model,2025-08-17 09:56:43.222236,1,0.714,0.733,0.709,0.376,0.328,0.240,4
1,all_data_20250817_095650.json,cookie,2025-08-17 09:56:43.749270,1,0.562,0.710,0.603,0.272,0.288,0.216,3
2,all_data_20250817_095650.json,client,2025-08-17 09:56:44.239768,1,0.468,0.714,0.349,0.288,0.312,0.224,2
3,all_data_20250817_095650.json,model,2025-08-17 09:56:45.642759,1,0.714,0.733,0.709,0.376,0.328,0.240,4
4,all_data_20250817_095650.json,model,2025-08-17 09:56:45.643461,2,0.714,0.733,0.709,0.376,0.328,0.240,4


In [68]:
import cirq

try:
    import qsimcirq
    HAVE_QSIM = True
except Exception:
    HAVE_QSIM = False

def make_circuit_from_row(row, n_qubits=6):
    v = np.array([row["quantum_host"], row["quantum_mate"], row["quantum_shared"]], dtype=float)
    v = np.clip(v, -1.0, 1.0)
    angles = (v + 1.0) * np.pi  # [0, 2π]

    qs = cirq.LineQubit.range(n_qubits)
    c = cirq.Circuit()

    for i, qb in enumerate(qs):
        a = angles[i % 3]
        c.append(cirq.ry(a)(qb))
        c.append(cirq.rz(a/2)(qb))
    for i in range(n_qubits-1):
        c.append(cirq.CNOT(qs[i], qs[i+1]))

    c.append(cirq.measure(*qs, key="m"))
    return c

def sample_bits(circuit, reps=256):
    sim = qsimcirq.QSimSimulator() if HAVE_QSIM else cirq.Simulator()
    res = sim.run(circuit, repetitions=reps)
    return res.measurements["m"].astype(np.uint8)

def bits_to_hist(bits):
    reps, n = bits.shape
    vals = (bits * (2 ** np.arange(n)[None, :])).sum(axis=1)
    hist = np.bincount(vals, minlength=2**n).astype(np.float32)
    hist /= max(1, hist.sum())
    return hist

from tqdm import tqdm

N_QUBITS = 6
REPS = 256
FEAT_DIM = 2**N_QUBITS

telemetry_feats = np.zeros((len(telemetry), FEAT_DIM), dtype=np.float32)
for i in tqdm(range(len(telemetry))):
    row = telemetry.iloc[i].to_dict()
    c = make_circuit_from_row(row, n_qubits=N_QUBITS)
    bits = sample_bits(c, reps=REPS)
    telemetry_feats[i] = bits_to_hist(bits)

print("telemetry_feats:", telemetry_feats.shape, "qsim:", HAVE_QSIM)


100%|██████████| 49/49 [00:00<00:00, 56.45it/s]

telemetry_feats: (49, 64) qsim: True


In [69]:
import os, json, math
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
import wave
import cv2

outdir = Path("/content/synth")
(outdir/"images").mkdir(parents=True, exist_ok=True)
(outdir/"audio").mkdir(parents=True, exist_ok=True)
(outdir/"video").mkdir(parents=True, exist_ok=True)
(outdir/"text").mkdir(parents=True, exist_ok=True)

def synth_text(row):
    # short “operator note” that encodes state
    ch, cm, cs = row["classical_host"], row["classical_mate"], row["classical_shared"]
    qh, qm, qs = row["quantum_host"], row["quantum_mate"], row["quantum_shared"]
    script = row["script_name"]
    cyc = int(row["cycle"])
    # simple narrative
    mood = "stable" if abs((qs - cs)) < 0.1 else ("drifting" if (qs < cs) else "amplifying")
    return (
        f"[{script}] cycle={cyc} mood={mood}. "
        f"classical(H={ch:.3f}, M={cm:.3f}, S={cs:.3f}) "
        f"quantum(H={qh:.3f}, M={qm:.3f}, S={qs:.3f})."
    )

def save_plot_image(row, path_png):
    ch, cm, cs = row["classical_host"], row["classical_mate"], row["classical_shared"]
    qh, qm, qs = row["quantum_host"], row["quantum_mate"], row["quantum_shared"]
    fig = plt.figure(figsize=(4, 3), dpi=120)
    ax = fig.add_subplot(111)
    ax.bar(["c_host","c_mate","c_shared","q_host","q_mate","q_shared"], [ch,cm,cs,qh,qm,qs])
    ax.set_ylim(0, 1)
    ax.set_title(f'{row["script_name"]} c{int(row["cycle"])}')
    fig.tight_layout()
    fig.savefig(path_png)
    plt.close(fig)

def save_audio_tone(row, path_wav, sr=16000, dur=1.0):
    # map Host/Mate/Shared to 3 tone freqs
    qh, qm, qs = float(row["quantum_host"]), float(row["quantum_mate"]), float(row["quantum_shared"])
    base = 220.0
    f1 = base * (1.0 + qh)
    f2 = base * (1.0 + qm) * 1.25
    f3 = base * (1.0 + qs) * 1.5
    t = np.linspace(0, dur, int(sr*dur), endpoint=False)
    sig = (0.33*np.sin(2*np.pi*f1*t) + 0.33*np.sin(2*np.pi*f2*t) + 0.33*np.sin(2*np.pi*f3*t))
    sig = sig / (np.max(np.abs(sig)) + 1e-9)
    pcm = (sig * 32767).astype(np.int16)

    with wave.open(str(path_wav), "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(pcm.tobytes())

def save_video_from_frames(row, path_mp4, n_frames=12, fps=12):
    # animate by slowly interpolating classical->quantum bars
    ch, cm, cs = float(row["classical_host"]), float(row["classical_mate"]), float(row["classical_shared"])
    qh, qm, qs = float(row["quantum_host"]), float(row["quantum_mate"]), float(row["quantum_shared"])

    W, H = 480, 360
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    vw = cv2.VideoWriter(str(path_mp4), fourcc, fps, (W, H))

    for k in range(n_frames):
        a = k/(n_frames-1)
        h = (1-a)*ch + a*qh
        m = (1-a)*cm + a*qm
        s = (1-a)*cs + a*qs

        # render simple bars with PIL
        img = Image.new("RGB", (W, H), (20, 20, 24))
        draw = ImageDraw.Draw(img)
        draw.text((12, 10), f'{row["script_name"]} c{int(row["cycle"])} a={a:.2f}', fill=(230,230,230))

        vals = [h, m, s]
        labels = ["host", "mate", "shared"]
        x0 = 60
        for i,(lab,val) in enumerate(zip(labels, vals)):
            x = x0 + i*120
            y_base = 320
            bar_h = int(240 * max(0.0, min(1.0, val)))
            draw.rectangle([x, y_base-bar_h, x+60, y_base], fill=(90, 170, 240))
            draw.text((x, y_base+8), lab, fill=(230,230,230))
            draw.text((x, y_base- bar_h - 18), f"{val:.2f}", fill=(230,230,230))

        frame = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
        vw.write(frame)

    vw.release()

manifest_path = outdir/"manifest.jsonl"
with open(manifest_path, "w") as f:
    for i in tqdm(range(len(telemetry))):
        row = telemetry.iloc[i].to_dict()
        ts = telemetry.iloc[i]["timestamp"].isoformat()
        script = row["script_name"]
        cyc = int(row["cycle"])

        # file naming
        stem = f"{script}_c{cyc:06d}_{i:07d}"
        txt_path = outdir/"text"/f"{stem}.txt"
        img_path = outdir/"images"/f"{stem}.png"
        wav_path = outdir/"audio"/f"{stem}.wav"
        mp4_path = outdir/"video"/f"{stem}.mp4"

        txt = synth_text(row)
        txt_path.write_text(txt, encoding="utf-8")
        save_plot_image(row, img_path)
        save_audio_tone(row, wav_path)
        save_video_from_frames(row, mp4_path)

        rec = {
            "i": i,
            "timestamp": ts,
            "script_name": script,
            "script_id": int(telemetry.iloc[i]["script_id"]),
            "cycle": cyc,
            "text": txt,
            "image_path": str(img_path),
            "audio_path": str(wav_path),
            "video_path": str(mp4_path)
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

100%|██████████| 49/49 [00:20<00:00,  2.39it/s]


In [70]:
import torch
from sentence_transformers import SentenceTransformer
from transformers import CLIPProcessor, CLIPModel, WhisperProcessor, WhisperModel
import librosa
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

text_teacher = SentenceTransformer("all-MiniLM-L6-v2")  # 384-d
text_teacher.eval()

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
clip_proc  = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

whisper_model = WhisperModel.from_pretrained("openai/whisper-small").to(DEVICE).eval()
whisper_proc  = WhisperProcessor.from_pretrained("openai/whisper-small")


In [71]:
import numpy as np
import cv2
from tqdm import tqdm

def embed_text(t: str):
    v = text_teacher.encode([t], normalize_embeddings=True)[0].astype(np.float32)
    return v  # (384,)

@torch.no_grad()
def embed_image(path: str):
    img = Image.open(path).convert("RGB")
    inputs = clip_proc(images=img, return_tensors="pt").to(DEVICE)
    feats = clip_model.get_image_features(**inputs)
    feats = torch.nn.functional.normalize(feats, dim=-1)
    return feats[0].cpu().numpy().astype(np.float32)  # (512,)

@torch.no_grad()
def embed_audio(path: str, sr=16000, max_sec=2):
    wav, _ = librosa.load(path, sr=sr, mono=True)
    wav = wav[: sr*max_sec]
    inputs = whisper_proc(wav, sampling_rate=sr, return_tensors="pt")
    input_features = inputs.input_features.to(DEVICE)
    enc = whisper_model.encoder(input_features).last_hidden_state  # (1,T,768)
    v = enc.mean(dim=1)[0]
    v = torch.nn.functional.normalize(v, dim=-1)
    return v.cpu().numpy().astype(np.float32)  # (768,)

@torch.no_grad()
def embed_video(path: str, n_frames=8):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        return np.zeros((512,), dtype=np.float32)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs = np.linspace(0, max(0, frame_count-1), n_frames).astype(int)
    want = set(idxs.tolist())

    embs = []
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i in want:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = Image.fromarray(frame)
            inputs = clip_proc(images=img, return_tensors="pt").to(DEVICE)
            feats = clip_model.get_image_features(**inputs)
            feats = torch.nn.functional.normalize(feats, dim=-1)
            embs.append(feats[0].cpu().numpy())
        i += 1
    cap.release()
    if not embs:
        return np.zeros((512,), dtype=np.float32)
    v = np.mean(np.stack(embs).astype(np.float32), axis=0)
    v = v / (np.linalg.norm(v) + 1e-9)
    return v.astype(np.float32)

# Read manifest
records = [json.loads(line) for line in open(manifest_path, "r", encoding="utf-8")]

D_TEXT, D_IMG, D_AUD, D_VID = 384, 512, 768, 512
E_text = np.zeros((len(records), D_TEXT), dtype=np.float32)
E_img  = np.zeros((len(records), D_IMG),  dtype=np.float32)
E_aud  = np.zeros((len(records), D_AUD),  dtype=np.float32)
E_vid  = np.zeros((len(records), D_VID),  dtype=np.float32)

for r in tqdm(records):
    i = r["i"]
    E_text[i] = embed_text(r["text"])
    E_img[i]  = embed_image(r["image_path"])
    E_aud[i]  = embed_audio(r["audio_path"])
    E_vid[i]  = embed_video(r["video_path"])

np.save(outdir/"E_text.npy", E_text)
np.save(outdir/"E_img.npy",  E_img)
np.save(outdir/"E_aud.npy",  E_aud)
np.save(outdir/"E_vid.npy",  E_vid)

print("cached embeddings saved in", outdir)


100%|██████████| 49/49 [08:45<00:00, 10.72s/it]

cached embeddings saved in /content/synth


In [72]:
from collections import defaultdict

targets = telemetry[["quantum_host","quantum_mate","quantum_shared"]].values.astype(np.float32)
script_ids = telemetry["script_id"].values.astype(np.int64)

# modality embeddings
E_text = np.load(outdir/"E_text.npy").astype(np.float32)
E_img  = np.load(outdir/"E_img.npy").astype(np.float32)
E_aud  = np.load(outdir/"E_aud.npy").astype(np.float32)
E_vid  = np.load(outdir/"E_vid.npy").astype(np.float32)

# reduce telemetry feature dim -> d_telem using PCA-ish linear projection (fast)
# (You can replace with nn.Linear in the model; we’ll keep it model-side to stay flexible.)
D_TELEM = telemetry_feats.shape[1]

by_script = defaultdict(list)
for idx, sid in enumerate(script_ids):
    by_script[int(sid)].append(idx)

SEQ_LEN = 5 # Changed from 32 to 5 to allow sequence creation

X_tel, X_txt, X_img, X_aud, X_vid, S_id = [], [], [], [], [], []
Y_next = []
T_txt, T_img, T_aud, T_vid = [], [], [], []

for sid, idxs in by_script.items():
    for j in range(0, len(idxs) - SEQ_LEN - 1):
        win = idxs[j:j+SEQ_LEN]
        nxt = idxs[j+SEQ_LEN]

        X_tel.append(telemetry_feats[win])   # (T, D_TELEM)
        X_txt.append(E_text[win])
        X_img.append(E_img[win])
        X_aud.append(E_aud[win])
        X_vid.append(E_vid[win])

        S_id.append(sid)
        Y_next.append(targets[nxt])

        # distill target = teacher embeddings at last timestep of window
        last = win[-1]
        T_txt.append(E_text[last]); T_img.append(E_img[last]); T_aud.append(E_aud[last]); T_vid.append(E_vid[last])

# Only stack if lists are not empty
if X_tel:
    X_tel = np.stack(X_tel).astype(np.float32)
    X_txt = np.stack(X_txt).astype(np.float32)
    X_img = np.stack(X_img).astype(np.float32)
    X_aud = np.stack(X_aud).astype(np.float32)
    X_vid = np.stack(X_vid).astype(np.float32)
    S_id  = np.array(S_id, dtype=np.int64)

    Y_next = np.stack(Y_next).astype(np.float32)
    T_txt  = np.stack(T_txt).astype(np.float32)
    T_img  = np.stack(T_img).astype(np.float32)
    T_aud  = np.stack(T_aud).astype(np.float32)
    T_vid  = np.stack(T_vid).astype(np.float32)

    print("seq dataset:", X_tel.shape, Y_next.shape, "scripts:", n_scripts)
else:
    print("No sequences could be created with the current SEQ_LEN and data.")
    X_tel, X_txt, X_img, X_aud, X_vid, S_id, Y_next, T_txt, T_img, T_aud, T_vid = (
        np.array([]) for _ in range(11)
    )



seq dataset: (28, 5, 64) (28, 3) scripts: 5


In [73]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class MMSeqDataset(Dataset):
    def __init__(self, X_tel, X_txt, X_img, X_aud, X_vid, S_id, Y_next, T_txt, T_img, T_aud, T_vid):
        self.X_tel = torch.from_numpy(X_tel)
        self.X_txt = torch.from_numpy(X_txt)
        self.X_img = torch.from_numpy(X_img)
        self.X_aud = torch.from_numpy(X_aud)
        self.X_vid = torch.from_numpy(X_vid)
        self.S_id  = torch.from_numpy(S_id)
        self.Y_next= torch.from_numpy(Y_next)
        self.T_txt = torch.from_numpy(T_txt)
        self.T_img = torch.from_numpy(T_img)
        self.T_aud = torch.from_numpy(T_aud)
        self.T_vid = torch.from_numpy(T_vid)

    def __len__(self): return self.X_tel.shape[0]
    def __getitem__(self, i):
        return (self.X_tel[i], self.X_txt[i], self.X_img[i], self.X_aud[i], self.X_vid[i],
                self.S_id[i], self.Y_next[i], self.T_txt[i], self.T_img[i], self.T_aud[i], self.T_vid[i])

def cosine_loss(a, b, eps=1e-8):
    a = a / (a.norm(dim=-1, keepdim=True) + eps)
    b = b / (b.norm(dim=-1, keepdim=True) + eps)
    return 1.0 - (a*b).sum(dim=-1).mean()

class MultiModalStudent(nn.Module):
    def __init__(self, d_telem, n_scripts, d_model=256, nhead=8, num_layers=4,
                 d_text=384, d_img=512, d_vid=512, d_aud=768, dropout=0.1):
        super().__init__()
        self.script_emb = nn.Embedding(n_scripts, d_model)

        self.p_tel  = nn.Linear(d_telem, d_model)
        self.p_text = nn.Linear(d_text, d_model)
        self.p_img  = nn.Linear(d_img, d_model)
        self.p_aud  = nn.Linear(d_aud, d_model)
        self.p_vid  = nn.Linear(d_vid, d_model)

        self.type_emb = nn.Embedding(5, d_model)  # 0=TEL 1=TXT 2=IMG 3=AUD 4=VID

        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4*d_model,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.tr = nn.TransformerEncoder(enc, num_layers=num_layers)

        self.next_q = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 3))

        self.dist_text = nn.Linear(d_model, d_text)
        self.dist_img  = nn.Linear(d_model, d_img)
        self.dist_aud  = nn.Linear(d_model, d_aud)
        self.dist_vid  = nn.Linear(d_model, d_vid)

    def forward(self, tel_seq, txt_seq, img_seq, aud_seq, vid_seq, script_id):
        B,T,_ = tel_seq.shape
        s = self.script_emb(script_id).unsqueeze(1).unsqueeze(1)  # (B,1,1,d)

        tel = self.p_tel(tel_seq)  + self.type_emb(torch.zeros((B,T),dtype=torch.long,device=tel_seq.device)) + s.squeeze(1)
        txt = self.p_text(txt_seq) + self.type_emb(torch.ones((B,T),dtype=torch.long,device=tel_seq.device)) + s.squeeze(1)
        img = self.p_img(img_seq)  + self.type_emb(torch.full((B,T),2,dtype=torch.long,device=tel_seq.device)) + s.squeeze(1)
        aud = self.p_aud(aud_seq)  + self.type_emb(torch.full((B,T),3,dtype=torch.long,device=tel_seq.device)) + s.squeeze(1)
        vid = self.p_vid(vid_seq)  + self.type_emb(torch.full((B,T),4,dtype=torch.long,device=tel_seq.device)) + s.squeeze(1)

        tokens = torch.stack([tel,txt,img,aud,vid], dim=2).reshape(B, T*5, -1)
        h = self.tr(tokens)

        summary = h[:, (T-1)*5 + 0, :]  # last timestep telemetry token
        next_q = self.next_q(summary)

        dist = {
            "text": self.dist_text(summary),
            "img":  self.dist_img(summary),
            "aud":  self.dist_aud(summary),
            "vid":  self.dist_vid(summary),
        }
        return next_q, dist

torch_device = "cuda" if torch.cuda.is_available() else "cpu"

idx = np.arange(len(X_tel))
tr_idx, va_idx = train_test_split(idx, test_size=0.2, random_state=42)

ds_tr = MMSeqDataset(X_tel[tr_idx], X_txt[tr_idx], X_img[tr_idx], X_aud[tr_idx], X_vid[tr_idx],
                    S_id[tr_idx], Y_next[tr_idx], T_txt[tr_idx], T_img[tr_idx], T_aud[tr_idx], T_vid[tr_idx])
ds_va = MMSeqDataset(X_tel[va_idx], X_txt[va_idx], X_img[va_idx], X_aud[va_idx], X_vid[va_idx],
                    S_id[va_idx], Y_next[va_idx], T_txt[va_idx], T_img[va_idx], T_aud[va_idx], T_vid[va_idx])

dl_tr = DataLoader(ds_tr, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
dl_va = DataLoader(ds_va, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

model = MultiModalStudent(d_telem=D_TELEM, n_scripts=n_scripts).to(torch_device)

opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
loss_next = nn.SmoothL1Loss()

w_txt, w_img, w_aud, w_vid = 0.2, 0.2, 0.2, 0.2

def eval_one():
    model.eval()
    tot = 0.0
    n = 0
    with torch.no_grad():
        for batch in dl_va:
            tel, txt, img, aud, vid, sid, y, tt, ti, ta, tv = batch
            tel, txt, img, aud, vid = tel.to(torch_device), txt.to(torch_device), img.to(torch_device), aud.to(torch_device), vid.to(torch_device)
            sid, y = sid.to(torch_device), y.to(torch_device)
            tt, ti, ta, tv = tt.to(torch_device), ti.to(torch_device), ta.to(torch_device), tv.to(torch_device)

            pred_q, dist = model(tel, txt, img, aud, vid, sid)
            L = loss_next(pred_q, y)
            L = L + w_txt*cosine_loss(dist["text"], tt)
            L = L + w_img*cosine_loss(dist["img"],  ti)
            L = L + w_aud*cosine_loss(dist["aud"],  ta)
            L = L + w_vid*cosine_loss(dist["vid"],  tv)

            tot += float(L) * tel.size(0)
            n += tel.size(0)
    return tot/n

for epoch in range(1, 101):
    model.train()
    tot = 0.0
    n = 0
    for batch in dl_tr:
        tel, txt, img, aud, vid, sid, y, tt, ti, ta, tv = batch
        tel, txt, img, aud, vid = tel.to(torch_device), txt.to(torch_device), img.to(torch_device), aud.to(torch_device), vid.to(torch_device)
        sid, y = sid.to(torch_device), y.to(torch_device)
        tt, ti, ta, tv = tt.to(torch_device), ti.to(torch_device), ta.to(torch_device), tv.to(torch_device)

        opt.zero_grad(set_to_none=True)
        pred_q, dist = model(tel, txt, img, aud, vid, sid)

        L = loss_next(pred_q, y)
        L = L + w_txt*cosine_loss(dist["text"], tt)
        L = L + w_img*cosine_loss(dist["img"],  ti)
        L = L + w_aud*cosine_loss(dist["aud"],  ta)
        L = L + w_vid*cosine_loss(dist["vid"],  tv)

        L.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        tot += float(L) * tel.size(0)
        n += tel.size(0)

    print(f"epoch {epoch:02d} train {tot/n:.5f}  val {eval_one():.5f}")

WARNING    /usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
 [py.warnings]
  warnings.warn(warn_msg)



epoch 01 train 0.91858  val 1.48877
epoch 02 train 1.43853  val 1.26643
epoch 03 train 1.23760  val 0.91216
epoch 04 train 0.92007  val 0.67227
epoch 05 train 0.68011  val 0.68290
epoch 06 train 0.68333  val 0.67033
epoch 07 train 0.67110  val 0.58770
epoch 08 train 0.59311  val 0.51256
epoch 09 train 0.51544  val 0.47431
epoch 10 train 0.47003  val 0.43329
epoch 11 train 0.43202  val 0.38077
epoch 12 train 0.38599  val 0.34745
epoch 13 train 0.35522  val 0.32504
epoch 14 train 0.33453  val 0.29806
epoch 15 train 0.30379  val 0.28187
epoch 16 train 0.28520  val 0.26373
epoch 17 train 0.26687  val 0.23853
epoch 18 train 0.24325  val 0.22136
epoch 19 train 0.22874  val 0.20829
epoch 20 train 0.21730  val 0.19336
epoch 21 train 0.20235  val 0.17828
epoch 22 train 0.18691  val 0.16990
epoch 23 train 0.17654  val 0.15524
epoch 24 train 0.16433  val 0.14325
epoch 25 train 0.15353  val 0.13381
epoch 26 train 0.14604  val 0.12239
epoch 27 train 0.13583  val 0.11182
epoch 28 train 0.12258  val 

In [74]:
from brian2 import *

def brian2_decode(pred_vec, duration_ms=200):
    v = np.array(pred_vec, dtype=float)
    v = np.clip(v, -1.0, 1.0)

    start_scope()
    defaultclock.dt = 0.1*ms
    N = 64
    tau = 10*ms
    eqs = '''
    dv/dt = (-v + I)/tau : 1
    I : 1
    '''
    G = NeuronGroup(N, eqs, threshold='v>1', reset='v=0', method='euler')

    base = 0.6
    weights = np.linspace(0.5, 1.5, N)
    drive = base + weights*(0.2*v[0] + 0.2*v[1] + 0.2*v[2])
    G.I = drive

    M = SpikeMonitor(G)
    run(duration_ms*ms)

    rate_hz = M.num_spikes / (N * (duration_ms/1000.0))
    return float(rate_hz), M.count[:]

# demo a single sample from val set
model.eval()
batch = next(iter(dl_va))
tel, txt, img, aud, vid, sid, y, tt, ti, ta, tv = batch
with torch.no_grad():
    pred_q, _ = model(tel[:1].to(torch_device), txt[:1].to(torch_device), img[:1].to(torch_device), aud[:1].to(torch_device), vid[:1].to(torch_device), sid[:1].to(torch_device))
pred = pred_q.cpu().numpy()[0]
rate_hz, counts = brian2_decode(pred, duration_ms=200)
pred, rate_hz

WARNING    'v' is an internal variable of group 'neurongroup', but also exists in the run namespace with the value array([0.38341156, 0.28158858, 0.31218296]). The internal variable will be used. [brian2.groups.group.Group.resolve.resolution_conflict]


(array([0.38341156, 0.28158858, 0.31218296], dtype=float32), 0.0)